# 03 — Results Analysis

Post-training evaluation: per-horizon metrics, time series overlays, error heatmaps.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt
from pathlib import Path

from src.data.dataset import get_dataloaders, load_graph, load_scaler
from src.models.gcn_lstm import GCNLSTM
from src.training.metrics import compute_all_metrics

%matplotlib inline

with open('../configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

In [ ]:
device = torch.device('cpu')
processed_dir = '../' + cfg['data']['processed_dir']

loaders = get_dataloaders(processed_dir, batch_size=cfg['data']['batch_size'], num_workers=0)
edge_index, edge_weight = load_graph(f'{processed_dir}/adj_mx.npz')
mean, std = load_scaler(f'{processed_dir}/scaler.npz')

model = GCNLSTM(
    in_channels=cfg['model']['in_channels'],
    hidden_dim=cfg['model']['hidden_dim'],
    out_horizon=cfg['model']['out_horizon'],
    num_nodes=cfg['model']['num_nodes'],
    dropout=cfg['model']['dropout'],
)
ckpt = torch.load(f"../{cfg['training']['checkpoint_dir']}/{cfg['training']['best_model']}",
                  map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded from epoch {ckpt["epoch"]}')

In [ ]:
# Run test inference
all_preds, all_targets = [], []
with torch.no_grad():
    for X, Y in loaders['test']:
        pred = model(X, edge_index, edge_weight)
        target = Y.permute(0, 2, 1)
        all_preds.append(pred.numpy())
        all_targets.append(target.numpy())

preds = np.concatenate(all_preds)
targets = np.concatenate(all_targets)

# Inverse transform
preds_orig = preds * std[None, :, None] + mean[None, :, None]
targets_orig = targets * std[None, :, None] + mean[None, :, None]

print(f'Predictions shape: {preds_orig.shape}')
overall = compute_all_metrics(preds_orig, targets_orig)
for k, v in overall.items():
    print(f'  {k}: {v:.3f}')

In [ ]:
# Per-horizon metrics
horizons = {0: '5 min', 2: '15 min', 5: '30 min', 11: '60 min'}
for h_idx, label in horizons.items():
    m = compute_all_metrics(preds_orig[:, :, h_idx], targets_orig[:, :, h_idx])
    print(f'{label:>8s}: MAE={m["MAE"]:.3f}, RMSE={m["RMSE"]:.3f}, MAPE={m["MAPE"]:.2f}%')

In [ ]:
# Time series overlay for sample sensors
n_steps = 288  # 1 day
for sensor_idx in [0, 50, 100]:
    fig, ax = plt.subplots(figsize=(14, 3))
    x = np.arange(n_steps) * 5 / 60  # hours
    ax.plot(x, targets_orig[:n_steps, sensor_idx, 0], label='Actual', alpha=0.8)
    ax.plot(x, preds_orig[:n_steps, sensor_idx, 0], label='Predicted', alpha=0.8)
    ax.set_xlabel('Hour'); ax.set_ylabel('Speed (mph)')
    ax.set_title(f'Sensor {sensor_idx} — 5 min horizon')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Per-sensor MAE heatmap
import pandas as pd

sensor_mae = np.abs(preds_orig[:, :, 0] - targets_orig[:, :, 0]).mean(axis=0)
locs = pd.read_csv('../data/raw/graph_sensor_locations.csv')
locs = locs[['sensor_id', 'latitude', 'longitude']]

fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(locs['longitude'], locs['latitude'], c=sensor_mae,
                cmap='RdYlGn_r', s=25, edgecolors='k', linewidths=0.3)
plt.colorbar(sc, ax=ax, label='MAE (mph)', shrink=0.7)
ax.ticklabel_format(useOffset=False)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Per-Sensor MAE (5 min horizon)')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
# Per-sensor MAE heatmaps across forecast horizons
horizons_map = {0: '5 min', 3: '20 min', 7: '40 min', 11: '60 min'}

# Compute MAE per sensor at each horizon
mae_by_horizon = {}
for h_idx, label in horizons_map.items():
    mae_by_horizon[label] = np.abs(preds_orig[:, :, h_idx] - targets_orig[:, :, h_idx]).mean(axis=0)

# Shared color scale across all panels
vmin = min(m.min() for m in mae_by_horizon.values())
vmax = max(m.max() for m in mae_by_horizon.values())

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
for ax, (label, sensor_mae) in zip(axes.flat, mae_by_horizon.items()):
    sc = ax.scatter(locs['longitude'], locs['latitude'], c=sensor_mae,
                    cmap='RdYlGn_r', s=25, edgecolors='k', linewidths=0.3,
                    vmin=vmin, vmax=vmax)
    ax.ticklabel_format(useOffset=False)
    ax.set_title(f'Per-Sensor MAE ({label} horizon)')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.set_aspect('equal')

fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.91, 0.15, 0.02, 0.7])
fig.colorbar(sc, cax=cbar_ax, label='MAE (mph)')
fig.suptitle('Spatial Error Distribution by Forecast Horizon', fontsize=14)
plt.show()